# Estruturas probabilísticas no Redis

Este notebook usa, num banco gratuito do Redis Cloud, as seis estruturas probabilísticas que o Redis oferece como tipo de dado: filtro de Bloom (`BF.*`), filtro de cuco (`CF.*`), *count-min sketch* (`CMS.*`), *top-k* (`TOPK.*`), HyperLogLog (`PF*`) e *t-digest* (`TDIGEST.*`). Em cada seção, a estrutura é criada com parâmetros derivados do erro tolerado, recebe o registro de chamadas sintético, responde à pergunta e tem a resposta comparada com a resposta exata calculada em Python.

O MinHash não existe como tipo de dado no Redis e está só no notebook de HyperLogLog e MinHash.

A conexão é lida da variável `REDIS_URL` do arquivo `.env` desta pasta (ver `README.md`).

In [1]:
import os
import random

import numpy as np
import redis
from dotenv import load_dotenv

load_dotenv()
r = redis.from_url(os.environ["REDIS_URL"], decode_responses=True)
print("PING:", r.ping())
print("versão do Redis:", r.info("server")["redis_version"])

PING: True
versão do Redis: 8.6.2


Todas as chaves deste notebook começam com `aula14:`. As que existirem de uma execução anterior são apagadas antes de começar.

In [2]:
antigas = list(r.scan_iter("aula14:*"))
if antigas:
    r.delete(*antigas)
print("chaves antigas apagadas:", len(antigas))

chaves antigas apagadas: 0


## Dados

A lista de bloqueio tem 50.000 números; o registro de chamadas tem 200.000 chamadas originadas por 20.000 números com distribuição de Zipf, e cada chamada tem uma duração em segundos, com distribuição log-normal (muitas chamadas curtas, poucas longas).

O volume é pequeno para caber no limite de memória do plano gratuito e no tempo de envio pela rede; o comportamento das estruturas não depende do volume.

In [3]:
random.seed(42)
ddds = [11, 12, 13, 19, 21, 27, 31, 41, 47, 48, 51, 61, 62, 71, 81, 85, 91, 92]

def numero_aleatorio():
    ddd = random.choice(ddds)
    assinante = random.randint(0, 99_999_999)
    return f"+55{ddd}9{assinante:08d}"

todos = set()
while len(todos) < 100_000:
    todos.add(numero_aleatorio())
todos = sorted(todos)
random.shuffle(todos)
lista_bloqueio = todos[:50_000]
outros = todos[50_000:]

gerador = np.random.default_rng(42)
originadores = todos[:20_000]
pesos = 1 / np.arange(1, len(originadores) + 1) ** 1.1
pesos = pesos / pesos.sum()
indices = gerador.choice(len(originadores), size=200_000, p=pesos)
chamadas = []
for i in indices:
    chamadas.append(originadores[i])
duracoes = gerador.lognormal(mean=4.0, sigma=1.0, size=200_000).round(1)

print(len(lista_bloqueio), "números na lista de bloqueio")
print(len(chamadas), "chamadas; duração mediana:", np.median(duracoes), "s")

50000 números na lista de bloqueio
200000 chamadas; duração mediana: 54.7 s


Cada comando enviado isoladamente paga a ida e a volta pela rede até o servidor. A carga usa os comandos de inserção múltipla e um *pipeline*, que envia vários comandos de uma vez e lê as respostas no fim.

In [4]:
def em_lotes(lista, tamanho):
    lotes = []
    for inicio in range(0, len(lista), tamanho):
        lotes.append(lista[inicio:inicio + tamanho])
    return lotes

## Filtro de Bloom: o número está na lista de bloqueio?

`BF.RESERVE chave taxa_de_erro capacidade` recebe os dois parâmetros do dimensionamento, $p$ e $n$, e calcula $m$ e $k$.

In [5]:
r.bf().reserve("aula14:bloom", 0.01, 50_000)

pipe = r.pipeline(transaction=False)
for lote in em_lotes(lista_bloqueio, 5_000):
    pipe.bf().madd("aula14:bloom", *lote)
pipe.execute()

info = r.bf().info("aula14:bloom")
print("capacidade:", info.capacity, " itens inseridos:", info.insertedNum)
print("memória da chave:", r.memory_usage("aula14:bloom"), "bytes")

capacidade: 50000  itens inseridos: 49962
memória da chave: 69056 bytes


O contador de itens inseridos fica um pouco abaixo de 50.000: `BF.MADD` devolve 0, e não conta a inserção, quando os $k$ bits do número já estavam todos ligados — ou seja, quando o número já era um falso positivo no momento da inserção. A memória da chave, cerca de 69 KB, corresponde aos $9{,}6 \times 50.000 / 8 \approx 60$ KB do vetor mais a estrutura interna.

In [6]:
respostas = []
for lote in em_lotes(lista_bloqueio, 5_000):
    respostas.extend(r.bf().mexists("aula14:bloom", *lote))
falsos_negativos = 0
for resposta in respostas:
    if resposta == 0:
        falsos_negativos = falsos_negativos + 1

respostas = []
for lote in em_lotes(outros, 5_000):
    respostas.extend(r.bf().mexists("aula14:bloom", *lote))
falsos_positivos = 0
for resposta in respostas:
    if resposta == 1:
        falsos_positivos = falsos_positivos + 1

print("falsos negativos:", falsos_negativos)
print("taxa de falso positivo medida:", round(falsos_positivos / len(outros) * 100, 3), "%  (configurada: 1%)")

falsos negativos: 0
taxa de falso positivo medida: 0.518 %  (configurada: 1%)


## Filtro de cuco: a lista que muda

`CF.RESERVE chave capacidade` cria a tabela; o Redis usa baldes de 2 posições por padrão e impressões digitais de 8 bits, o que dá uma taxa de falso positivo prevista de cerca de $2b/2^f = 4/256 \approx 1{,}6\%$ com os baldes cheios.

In [7]:
r.cf().reserve("aula14:cuco", 60_000)

pipe = r.pipeline(transaction=False)
for numero in lista_bloqueio:
    pipe.cf().add("aula14:cuco", numero)
pipe.execute()
print("memória da chave:", r.memory_usage("aula14:cuco"), "bytes")

saindo = lista_bloqueio[:1_000]
print("antes: presentes entre os que vão sair =", sum(r.cf().mexists("aula14:cuco", *saindo)))

pipe = r.pipeline(transaction=False)
for numero in saindo:
    pipe.cf().delete("aula14:cuco", numero)
pipe.execute()
print("depois: presentes entre os que saíram =", sum(r.cf().mexists("aula14:cuco", *saindo)))

respostas = []
for lote in em_lotes(lista_bloqueio[1_000:], 5_000):
    respostas.extend(r.cf().mexists("aula14:cuco", *lote))
print("falsos negativos entre os que ficaram:", len(respostas) - sum(respostas))

respostas = []
for lote in em_lotes(outros, 5_000):
    respostas.extend(r.cf().mexists("aula14:cuco", *lote))
print("taxa de falso positivo medida:", round(sum(respostas) / len(outros) * 100, 3), "%")

memória da chave: 131176 bytes
antes: presentes entre os que vão sair = 1000


depois: presentes entre os que saíram = 10


falsos negativos entre os que ficaram: 0


taxa de falso positivo medida: 1.16 %


## *Count-min sketch*: quantas chamadas originou cada número?

`CMS.INITBYPROB chave ε δ` deriva as dimensões do erro tolerado. O Redis usa $w = \lceil 2/\varepsilon \rceil$ e $d = \lceil \log_2(1/\delta) \rceil$, e não as fórmulas de Cormode e Muthukrishnan ($e/\varepsilon$ e $\ln(1/\delta)$): o argumento é o mesmo, com a desigualdade de Markov aplicada para probabilidade $1/2$ por linha em vez de $1/e$, o que dá matriz mais estreita e mais funda para a mesma garantia. Com $\varepsilon = 0{,}001$ e $N = 200.000$, o erro fica abaixo de $\varepsilon N = 200$ chamadas com probabilidade de 99%.

In [8]:
r.cms().initbyprob("aula14:cms", 0.001, 0.01)

pipe = r.pipeline(transaction=False)
for lote in em_lotes(chamadas, 5_000):
    pipe.cms().incrby("aula14:cms", lote, [1] * len(lote))
pipe.execute()

info = r.cms().info("aula14:cms")
print("largura w:", info.width, " profundidade d:", info.depth, " total N:", info.count)
print("memória da chave:", r.memory_usage("aula14:cms"), "bytes")

largura w: 2000  profundidade d: 7  total N: 200000
memória da chave: 56064 bytes


In [9]:
contagem_exata = {}
for origem in chamadas:
    if origem not in contagem_exata:
        contagem_exata[origem] = 0
    contagem_exata[origem] = contagem_exata[origem] + 1

def pelo_total(par):
    return par[1]

ranking = sorted(contagem_exata.items(), key=pelo_total, reverse=True)
numeros_ranking = []
for numero, total in ranking:
    numeros_ranking.append(numero)
estimativas = []
for lote in em_lotes(numeros_ranking, 5_000):
    estimativas.extend(r.cms().query("aula14:cms", *lote))

abaixo = 0
acima_do_limite = 0
for i in range(len(ranking)):
    real = ranking[i][1]
    if estimativas[i] < real:
        abaixo = abaixo + 1
    if estimativas[i] > real + 0.001 * len(chamadas):
        acima_do_limite = acima_do_limite + 1
print("estimativas abaixo do real:", abaixo)
print("estimativas acima de real + epsilon * N:", acima_do_limite, "de", len(ranking))

print(f"{'posição':>8} {'real':>7} {'estimado':>9}")
for posicao in [1, 10, 100, 1000, 10000]:
    print(f"{posicao:>8} {ranking[posicao - 1][1]:>7} {estimativas[posicao - 1]:>9}")

estimativas abaixo do real: 0
estimativas acima de real + epsilon * N: 0 de 14267
 posição    real  estimado
       1   28971     28976
      10    2185      2196
     100     188       198
    1000      15        24
   10000       1        14


## *Top-k*: quais números mais originaram chamadas?

`TOPK.RESERVE chave k largura profundidade decaimento` cria o *HeavyKeeper* com $k = 10$.

In [10]:
r.topk().reserve("aula14:topk", 10, 2_000, 7, 0.925)

pipe = r.pipeline(transaction=False)
for lote in em_lotes(chamadas, 5_000):
    pipe.topk().add("aula14:topk", *lote)
pipe.execute()

lista = r.topk().list("aula14:topk", withcount=True)

# O redis-py converte para número toda resposta que parece número, e "+5511..." vira 5511...;
# o "+" é recolocado para comparar com a contagem exata.
top_redis = []
print(f"{'número':>16} {'estimado':>9} {'real':>7}")
for i in range(0, len(lista), 2):
    numero = str(lista[i])
    if not numero.startswith("+"):
        numero = "+" + numero
    top_redis.append(numero)
    print(f"{numero:>16} {lista[i + 1]:>9} {contagem_exata[numero]:>7}")

encontrados = 0
for numero, total in ranking[:10]:
    if numero in top_redis:
        encontrados = encontrados + 1
print("dos 10 maiores reais, presentes na lista:", encontrados)

          número  estimado    real
  +5551904246996     28971   28971
  +5513929222844     13748   13748
  +5585953458116      8799    8799
  +5519957565309      6423    6423
  +5571928945913      4825    4825
  +5562965538318      3977    3977
  +5591990271492      3419    3419
  +5531933772321      2980    2980
  +5562973987846      2577    2577
  +5519990784917      2185    2185
dos 10 maiores reais, presentes na lista: 10


## HyperLogLog: quantos números distintos originaram chamadas?

O HyperLogLog é tipo nativo do Redis, sem parâmetro: todo *sketch* tem 16.384 registradores de 6 bits, erro padrão de 0,81%. As chamadas são divididas por DDD de origem para mostrar a fusão.

In [11]:
pipe = r.pipeline(transaction=False)
for lote in em_lotes(chamadas, 5_000):
    por_ddd = {}
    for origem in lote:
        ddd = origem[3:5]
        if ddd not in por_ddd:
            por_ddd[ddd] = []
        por_ddd[ddd].append(origem)
    for ddd, numeros in por_ddd.items():
        pipe.pfadd("aula14:hll:" + ddd, *numeros)
pipe.execute()

print("distintos exatos:  ", len(set(chamadas)))
chaves_ddd = []
for ddd in ddds:
    chaves_ddd.append("aula14:hll:" + str(ddd))
r.pfmerge("aula14:hll:brasil", *chaves_ddd)
print("estimativa da fusão:", r.pfcount("aula14:hll:brasil"))
print("memória de um sketch:", r.memory_usage("aula14:hll:brasil"), "bytes")

print(f"{'DDD':>4} {'exato':>7} {'estimado':>9}")
for ddd in ["11", "21", "61"]:
    exato = set()
    for origem in chamadas:
        if origem[3:5] == ddd:
            exato.add(origem)
    print(f"{ddd:>4} {len(exato):>7} {r.pfcount('aula14:hll:' + ddd):>9}")

distintos exatos:   14267


estimativa da fusão: 14216
memória de um sketch: 14376 bytes
 DDD   exato  estimado


  11     743       742
  21     818       817


  61     803       804


Com menos de alguns milhares de distintos, o Redis guarda o *sketch* numa representação esparsa, menor que 12 KB e com estimativa quase exata; ao crescer, converte para a representação densa de 16.384 registradores.

## *t-digest*: qual o p99 da duração das chamadas?

`TDIGEST.CREATE chave COMPRESSION δ` limita o número de centróides; as estimativas são comparadas com `numpy.percentile`.

In [12]:
r.tdigest().create("aula14:duracao", compression=100)

pipe = r.pipeline(transaction=False)
# tolist() converte para float do Python; o redis-py envia o float do numpy com o nome do tipo junto
for lote in em_lotes(duracoes.tolist(), 5_000):
    pipe.tdigest().add("aula14:duracao", lote)
pipe.execute()
print("memória da chave:", r.memory_usage("aula14:duracao"), "bytes, contra", duracoes.nbytes, "bytes das durações")

quantis = [0.5, 0.9, 0.99, 0.999]
estimados = r.tdigest().quantile("aula14:duracao", *quantis)
print(f"{'quantil':>8} {'exato':>9} {'t-digest':>9}")
for i in range(len(quantis)):
    exato = np.percentile(duracoes, quantis[i] * 100)
    print(f"{quantis[i]:>8} {exato:>9.1f} {float(estimados[i]):>9.1f}")

print("mínimo:", r.tdigest().min("aula14:duracao"), " máximo:", r.tdigest().max("aula14:duracao"), " (exatos:", duracoes.min(), duracoes.max(), ")")
curtas = r.tdigest().cdf("aula14:duracao", 5)
print("fração de chamadas com até 5 s: t-digest", round(float(curtas[0]), 4), " exato", round(float((duracoes <= 5).mean()), 4))

memória da chave: 9880 bytes, contra 1600000 bytes das durações
 quantil     exato  t-digest
     0.5      54.7      56.0
     0.9     196.5     198.1
    0.99     558.6     561.9
   0.999    1215.9    1212.0


mínimo: 0.4  máximo: 5843.2  (exatos: 0.4 5843.2 )
fração de chamadas com até 5 s: t-digest 0.0081  exato 0.0085


## Limpeza

As chaves do notebook são apagadas para liberar a memória do banco gratuito.

In [13]:
chaves = list(r.scan_iter("aula14:*"))
r.delete(*chaves)
print("chaves apagadas:", len(chaves))

chaves apagadas: 24
